Data Processing for Products

In [0]:
from pyspark.sql.functions import * 

In [0]:
%run /Workspace/consolidated_pipeline/02_dimension_data_processing/utilities

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','products','Data Source')

catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')


In [0]:
base_path=f'/Volumes/{catalog}/{bronze_schema}/sports_volume/{data_source}.csv'

In [0]:
df = spark.read.format('csv') \
    .option('header','true') \
    .option('inferSchema','true') \
    .load(base_path) \
    .withColumn('read_time_stamp', current_timestamp())

display(df)

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

data cleaning in silver layer

In [0]:
df_bronze=spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')
display(df_bronze.limit(10))

Drop Duplicates

In [0]:
df_bronze.groupBy('product_id').count().filter(col('count')>1).show()

In [0]:
df_silver=df_bronze.dropDuplicates(['product_id'])

In [0]:
display(df_silver)

Title case fixes

In [0]:
df_silver=df_silver.withColumn('category',when(col('category').isNull(),None).otherwise(initcap(col('category'))))

Fix Spelling Mistake for Protien

In [0]:
df_silver=df_silver.withColumn('product_name',regexp_replace(col("product_name"),"(?i)Protein",'Protien')).\
    withColumn('category',regexp_replace(col("category"),"(?i)Protein",'Protien'))

In [0]:
df_silver=df_silver.withColumn('division',
                     when(col('category')=='Energy Bars','Nutrition Bars').
                     when(col('category')=='Protien Bars','Nutrition Bars').
                     when(col('category')=='Granola & Cereals','Breakfast Foods').
                     when(col('category')=='Recovery Dairy','Dairy & Recovery').
                     when(col('category')=='Healthy Snacks','Healthy Snacks').
                     when(col('category')=='Electrolyte Mix','Hydration & Electrolytes').
                     otherwise("Others")
                     )
df_silver=df_silver.withColumn('variant',regexp_extract('product_name',r'\((.*?)\)',1))
display(df_silver)


In [0]:
# create deterministic product_code

df_silver=df_silver.withColumn('product_code',
                     sha2(col('product_name').cast('string'),256)
                     ).\
                         withColumn('product_id',
                                    when(
                                        col('product_id').cast('string').rlike("^[0-9]+$"),
                                        col('product_id').cast('string')
                                        ).otherwise(
                                            lit(999999).cast('string')
                                    )
                                    ).withColumnRenamed('product_name','product')
display(df_silver)

In [0]:
df_silver.write.format('delta').option('delta.enableChangeDataFeed','true').option('mergeSchema','true').mode('overwrite').saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

In [0]:
df_silver=spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
df_gold=df_silver.select('product_code','product_id','division','category','product','variant')
display(df_gold)

In [0]:
df_gold.write.format('delta').option('mergeSchema','true').option('delta.enableChangeDataFeed','true').\
    mode('overwrite').saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
from delta.tables import DeltaTable
delta_table=DeltaTable.forName(spark,'fmcg.gold.dim_products')
df_child_products=spark.sql(f"select product_code,division,category,product,variant from fmcg.gold.sb_dim_products;")
df_child_products.show()

In [0]:
delta_table.alias('target').merge(
    source=df_child_products.alias('source'),
    condition='target.product_code=source.product_code'
).whenMatchedUpdate(set={
    'division':'source.division',
    'category':'source.category',
    'product':'source.product',
    'variant':'source.variant',

}).whenNotMatchedInsert(values={
    'product_code':'source.product_code',
    'division':'source.division',
    'category':'source.category',
    'product':'source.product',
    'variant':'source.variant',

}).execute()


In [0]:
spark.table("fmcg.gold.dim_products").count()